# Explore the USA model: where it does well, where it doesn't, and why

Every scored 2025 decision across the lower 48 + DC: a county-hour, **2 hours ahead**, *will a new outage start?*
Five models' chances side by side, plus what each decision saw. Built by `pixi run python -m ml.explore_data`
(rebuild it after the pipeline reruns `forecast_test` / `forecast_ensemble`). Reads only the parquet files in
`data/explore/`, never the warehouse, so it never blocks `pixi run build`.

**Three things to keep in mind while reading the numbers**

1. **Compare ratios, not raw PR-AUC.** PR-AUC rises with how common outages are. A stormy county scores higher
   than a quiet one for every model. The fair question is *how much better than warnings + outlook on the same
   rows* (`ratio`), or *what share of new outages got an alert* (`caught`).
2. **Small groups are noise.** A county with 8 new outages in a year can show any ratio. The helpers drop groups
   below `min_outages` (default 30); the 95% ranges on the headline are about Â±15% even for the whole country.
3. **`caught` uses one country-wide alert level:** the top 0.75% of 2025's decisions for that model (about one
   hour in 130), the share the site quotes. A county "catches" an outage when its chance was above that level.

`win_*` columns are the window ahead's weather *as it happened* (hindsight). They're for describing the
situation, e.g. "was it really a thunderstorm?"; the headline model only saw forecasts of them.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import duckdb
import geopandas as gpd
import pandas as pd
import plotly.express as px
from scipy.stats import spearmanr
from sklearn.metrics import average_precision_score as ap

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

con = duckdb.connect()                       # in memory; reads data/explore/*.parquet
con.sql("SET memory_limit = '24GB'")
con.sql("CREATE VIEW d AS SELECT * FROM 'data/explore/decisions_2025.parquet'")   # one row per decision
con.sql("CREATE VIEW c AS SELECT * FROM 'data/explore/counties_2025.parquet'")    # one row per county


def q(sql):
    """Any SQL over d (decisions) and c (counties) -> DataFrame.  q("SELECT * FROM d LIMIT 5")"""
    return con.sql(sql).df()


MODELS = {
    "p_official": "Warnings + outlook",
    "p_no_forecast": "No forecasts",
    "p_newest": "HRRR, newest run",
    "p_mean": "HRRR, mean of 5 runs (headline)",
    "p_perfect": "Perfect weather (ceiling)",
}
SHOW = ("p_official", "p_no_forecast", "p_mean")     # the default three in tables
ALERT_SHARE = 0.0075
LEVEL = q("SELECT " + ", ".join(f"quantile_disc({k}, {1 - ALERT_SHARE}) AS {k}" for k in MODELS) + " FROM d"
          ).iloc[0].to_dict()
print("alert level per model (chance above which an hour gets an alert):", {k: round(v, 4) for k, v in LEVEL.items()})

In [ ]:
def _scores(g, models):
    y = g.y.to_numpy()
    out = {"decisions": len(y), "new_outages": int(y.sum()), "base_rate_pct": 100 * y.mean()}
    if not y.any():
        return out
    for k in models:
        p = g[k].to_numpy()
        out[f"pr_auc {k}"] = ap(y, p)
        out[f"caught {k}"] = (p[y == 1] >= LEVEL[k]).mean()
    if "p_official" in models:
        for k in models:
            if k != "p_official":
                out[f"ratio {k}"] = out[f"pr_auc {k}"] / out["pr_auc p_official"]
    return out


def score(where="TRUE", models=SHOW):
    """Every model's PR-AUC, ratio to warnings + outlook and share caught on the decisions matching `where` (SQL)."""
    g = q(f"SELECT y, {', '.join(models)} FROM d WHERE {where}")
    return pd.Series(_scores(g, models), name=where).to_frame()


def score_by(by, where="TRUE", models=SHOW, min_outages=30):
    """score() for each value of `by` (any SQL expression over d), groups with at least `min_outages` new outages."""
    g = q(f"SELECT {by} AS grp, y, {', '.join(models)} FROM d WHERE {where}")
    out = pd.DataFrame([{"group": k, **_scores(x, models)} for k, x in g.groupby("grp", sort=True)])
    return out[out.new_outages >= min_outages].set_index("group")


geo = gpd.read_parquet("data/raw/geo/regions_study.parquet")[["region_fips", "geometry"]].to_crs(4326)
geo["geometry"] = geo.geometry.simplify(0.01)
GEOJSON = geo.__geo_interface__


def county_map(col, where="new_outages >= 20", title=None, **kw):
    """A county map of any column of c (or SQL expression `AS value`), counties matching `where`. Extra arguments
    go to px.choropleth (color_continuous_scale=, range_color=, ...)."""
    df = q(f"SELECT region_fips, region, state, new_outages, {col} AS value FROM c WHERE {where}")
    fig = px.choropleth(df, geojson=GEOJSON, locations="region_fips", featureidkey="properties.region_fips",
                        color="value", scope="usa", hover_name="region",
                        hover_data={"state": True, "new_outages": True, "region_fips": False},
                        title=title or col, **kw)
    fig.update_traces(marker_line_width=0)
    fig.update_layout(height=520, margin=dict(l=0, r=0, t=40, b=0))
    return fig

## 1. The headline, recomputed

Should match `ml/results/forecast_ensemble_2025.csv` (mean of 5 runs 0.0868) and `forecast_test_2025.csv`
(warnings + outlook 0.0216).

In [ ]:
score(models=tuple(MODELS))

## 2. Where: state by state

`ratio p_mean` above 1 means the headline model beats warnings + outlook in that state. Sort by any column.

In [ ]:
states = score_by("state")
states.sort_values("ratio p_mean")

In [ ]:
px.bar(states.reset_index().sort_values("ratio p_mean"), x="group", y="ratio p_mean", hover_data=["new_outages"],
       log_y=True, title="Headline model vs warnings + outlook, by state (PR-AUC ratio, log scale)",
       labels={"group": "state"}).add_hline(y=1, line_dash="dot")

## 3. Where: county by county

The log2 of the ratio: 0 = no better than warnings + outlook, 1 = twice as good, 2 = 4Ã—. Counties with at
least 20 new outages; hover for the name. Try `county_map("caught_p_mean")`, `county_map("base_rate * 100")`,
or any place column (`county_map("canopy_mean_pct", where="TRUE")`).

In [ ]:
county_map("log2(ratio_p_mean)", color_continuous_scale="RdBu", range_color=(-3, 3),
           title="Headline model vs warnings + outlook (log2 PR-AUC ratio; blue = model better)")

In [ ]:
county_map("caught_p_mean - caught_p_official", color_continuous_scale="RdBu", range_color=(-0.5, 0.5),
           title="Share of new outages with an alert: headline model minus warnings + outlook")

## 4. When: month, hour of day, storm day

`hour_utc` is UTC (US Central = UTC âˆ’ 6 in winter, âˆ’ 5 in summer).

In [ ]:
score_by("month")

In [ ]:
score_by("hour_utc")

In [ ]:
days = q(f"""
    SELECT storm_day, sum(y) AS new_outages, count(DISTINCT CASE WHEN y = 1 THEN state END) AS states,
           avg(CASE WHEN y = 1 THEN (p_mean >= {LEVEL['p_mean']})::INT END) AS caught_model,
           avg(CASE WHEN y = 1 THEN (p_official >= {LEVEL['p_official']})::INT END) AS caught_official
    FROM d GROUP BY ALL HAVING sum(y) >= 50 ORDER BY new_outages DESC""")
px.scatter(days, x="caught_official", y="caught_model", size="new_outages", hover_data=["storm_day", "states"],
           title="Each storm day (12 UTC to 12 UTC, 50+ new outages): share caught, model vs warnings + outlook"
           ).add_shape(type="line", x0=0, y0=0, x1=1, y1=1, line_dash="dot")

In [ ]:
# big storm days where warnings + outlook caught more than the model: where to look first
# (days with outages spread thinly across many states and nothing caught are background, not storms)
days.assign(model_minus_official=days.caught_model - days.caught_official
            )[days.new_outages >= 200].sort_values("model_minus_official").head(10)

## 5. What kind of weather

A rough label per decision from the window's weather as it happened plus the warnings. Change the rules and
rerun. Winter storms and wind without thunderstorms are where warnings are weakest, so expect the biggest ratios
there; calm hours with outages are mostly not weather (equipment, heat, planned work) and nothing can predict them.

In [ ]:
STORM_TYPE = """
CASE WHEN warn_winter::INT > 0 OR coalesce(win_icing_hours, 0) > 0 OR coalesce(icing_hours, 0) > 0 THEN 'winter / ice'
     WHEN warn_tor::INT > 0 OR warn_svr::INT > 0 OR coalesce(win_cape_max_jkg, 0) >= 1000 THEN 'thunderstorm'
     WHEN coalesce(win_gust_max_ms, 0) >= 15 THEN 'wind, no thunderstorm'
     WHEN coalesce(win_gust_max_ms, 0) < 10 AND coalesce(win_precip_max_1h_mm, 0) < 1 THEN 'calm'
     ELSE 'other' END"""
score_by(STORM_TYPE, models=tuple(MODELS))

In [ ]:
# how strong the wind gets in the window ahead (m/s; 15 m/s = 34 mph, 25 m/s = 56 mph)
score_by("least(floor(win_gust_max_ms / 5) * 5, 35)")

## 6. Misses and false alarms: what's different about them?

Among new outages: the ones the headline model caught vs missed. Among alerts: true vs false. Medians of what the
decision saw. If missed outages have calm weather and no recent outages, they're the unpredictable kind; if
they have strong wind the model under-rated, that's something to fix.

In [ ]:
FEATS = ["gust_max_ms", "win_gust_max_ms", "win_cape_max_jkg", "win_precip_max_1h_mm", "win_temp_min_c",
         "pct_out_now", "max_pct_out_prev_24h", "spc_cat", "customers", "outcome_peak_pct"]
med = ", ".join(f"median({f}) AS {f}" for f in FEATS)
L = LEVEL["p_mean"]
pd.concat([
    q(f"SELECT CASE WHEN p_mean >= {L} THEN 'outage, caught' ELSE 'outage, missed' END AS kind, count(*) AS n, "
      f"{med}, avg((warn_svr::INT + warn_tor::INT + warn_wind::INT + warn_winter::INT > 0)::INT) AS share_warned "
      f"FROM d WHERE y = 1 GROUP BY 1"),
    q(f"SELECT CASE WHEN y = 1 THEN 'alert, true' ELSE 'alert, false alarm' END AS kind, count(*) AS n, "
      f"{med}, avg((warn_svr::INT + warn_tor::INT + warn_wind::INT + warn_winter::INT > 0)::INT) AS share_warned "
      f"FROM d WHERE p_mean >= {L} GROUP BY 1"),
]).set_index("kind").T

In [ ]:
# the biggest outages (customers out) the headline model missed, one row per county and storm day (its first
# missed hour). Check the top ones against the news: a county "100% out" in calm weather is often a utility's
# reporting glitch in EAGLE-I, not a real outage (e.g. Nassau + Suffolk, NY, 2 Feb 2025)
q(f"""SELECT state, county, storm_day, block_start, round(outcome_peak_pct, 1) AS peak_pct_out, customers,
             p_mean, p_official, gust_max_ms, win_gust_max_ms, win_cape_max_jkg, spc_cat, warn_svr, warn_wind
      FROM d WHERE y = 1 AND p_mean < {L}
      QUALIFY row_number() OVER (PARTITION BY region_fips, storm_day ORDER BY block_start) = 1
      ORDER BY outcome_peak_pct * customers DESC LIMIT 20""")

In [ ]:
# the strongest false alarms: high chance, no new outage
q("""SELECT state, county, block_start, p_mean, p_official, pct_out_now, max_pct_out_prev_24h,
             gust_max_ms, win_gust_max_ms, win_cape_max_jkg, spc_cat
      FROM d WHERE y = 0 ORDER BY p_mean DESC LIMIT 20""")

## 7. Which county traits go with doing well or badly?

Spearman correlation between a county's place data and the model's skill there (counties with 30+ new outages).
**Correlation, not cause:** stormier counties have more outages, which itself changes the ratio, and many traits
move together (rural â†” co-op â†” dense trees). Use this to pick what to look at, then check it with section 8.

In [ ]:
cty = q("SELECT * FROM c WHERE new_outages >= 30")
skip = {"region_fips", "region", "state", "decisions"} | {x for x in cty if x.startswith(("pr_auc_", "caught_",
        "ratio_", "alert_hours_"))}
factors = [x for x in cty.select_dtypes("number").columns if x not in skip]
corr = pd.DataFrame([{"factor": f, "n": int(cty[f].notna().sum()),
                      "rho_ratio": spearmanr(cty[f], cty.ratio_p_mean, nan_policy="omit").statistic,
                      "rho_caught": spearmanr(cty[f], cty.caught_p_mean, nan_policy="omit").statistic}
                     for f in factors if cty[f].nunique() > 2])
corr.reindex(corr.rho_ratio.abs().sort_values(ascending=False).index).head(25)

In [ ]:
TRAIT = "canopy_mean_pct"     # try any factor from the table above
px.scatter(cty, x=TRAIT, y="ratio_p_mean", size="new_outages", color="state", log_y=True, hover_name="region",
           title=f"{TRAIT} vs the headline model's ratio to warnings + outlook (one dot per county)"
           ).add_hline(y=1, line_dash="dot").update_layout(showlegend=False)

## 8. Playground

`score(where)` and `score_by(by, where)` take any SQL over the decision columns (`q("DESCRIBE d")` lists them).

In [ ]:
score("state = 'Texas' AND month BETWEEN 5 AND 8")     # states are full names

In [ ]:
score_by("state", where="warn_winter::INT > 0", min_outages=20)

In [ ]:
# counties with lots of outages where the model is no better than warnings + outlook
q("""SELECT state, region, new_outages, pr_auc_p_official, pr_auc_p_mean, ratio_p_mean, caught_p_mean,
            util_share_coop, canopy_mean_pct, customers
     FROM c WHERE new_outages >= 50 ORDER BY ratio_p_mean LIMIT 15""")

In [ ]:
# one county through its worst day of 2025: chances vs what happened (red lines = hours a new outage started).
# Change FIPS to any county: q("SELECT region_fips, region, state FROM c WHERE region LIKE 'Harris%'")
FIPS = q("SELECT region_fips FROM c ORDER BY new_outages DESC LIMIT 1").region_fips[0]
day = q(f"SELECT storm_day FROM d WHERE region_fips = '{FIPS}' GROUP BY 1 ORDER BY sum(y) DESC, 1 LIMIT 1").storm_day[0]
one = q(f"""SELECT block_start, y, p_mean, p_official, p_low, p_high, pct_out_now, gust_max_ms
            FROM d WHERE region_fips = '{FIPS}' AND storm_day BETWEEN DATE '{day}' - 1 AND DATE '{day}' + 1
            ORDER BY block_start""")
fig = px.line(one, x="block_start", y=["p_mean", "p_official"], title=f"{FIPS} around {day}")
for t in one.block_start[one.y == 1]:
    fig.add_vline(x=t, line_color="red", opacity=0.3)
fig

In [ ]:
con.close()   # run before rebuilding data/explore